# U03.LAB08 + U03.LAB09 / U03.E05 · Sistemas recomendadores reproducibles

**Asignatura:** INF-8239 Ciencia de Datos II
**Unidad:** 03 · Algoritmos y características generativas
**Código:** U03.LAB08 + U03.LAB09 (evidencia para U03.E05)

Consolida en un solo producto ejecutable: baseline de popularidad suavizada y recomendador por
contenido (LAB08), y factorización matricial con corte temporal, sistema híbrido y evaluación de
tres perfiles de usuario (LAB09). Reutiliza `src/inf8239_u03/` — no duplica lógica de
`scripts/lab08_content.py` ni `scripts/lab09_hybrid.py`.

Requiere el dataset MovieLens descargado (`uv run python scripts/download_data.py`).

In [1]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import json
import platform
import random
from datetime import datetime
from time import perf_counter

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

from inf8239_u03.config import MOVIELENS_DIR, ROOT
from inf8239_u03.data import load_movielens
from inf8239_u03.metrics import catalog_coverage, hit_rate_at_k
from inf8239_u03.recommenders import (
    ContentRecommender,
    MatrixFactorization,
    temporal_leave_one_out,
    weighted_popularity,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

reports = ROOT / "reports"
reports.mkdir(exist_ok=True)

print("pandas:", pd.__version__, " scikit-learn:", sklearn.__version__)

pandas: 2.3.3  scikit-learn: 1.9.1


## Paso 1 · Cargar y auditar MovieLens

Requiere haber corrido antes `uv run python scripts/download_data.py`. `load_movielens` valida el
contrato de columnas, claves foráneas y rango de ratings — el mismo contrato que usan las
pruebas automatizadas.

In [2]:
ratings, movies = load_movielens(MOVIELENS_DIR)
possible = ratings["userId"].nunique() * movies["movieId"].nunique()

print("Ratings:", ratings.shape, " Movies:", movies.shape)
print("Usuarios:", ratings["userId"].nunique(), " Items valorados:", ratings["movieId"].nunique())
print("Rango temporal:", ratings["timestamp"].min(), "-", ratings["timestamp"].max())
print("Densidad (observado/posible):", len(ratings) / possible)
ratings["rating"].describe()

Ratings: (100836, 4)  Movies: (9742, 3)
Usuarios: 610  Items valorados: 9724
Rango temporal: 828124615 - 1537799250
Densidad (observado/posible): 0.016968273253211548


count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

## Paso 2 · Baseline de popularidad suavizada

El promedio suavizado combina la media de cada película con la media global, ponderando por la
cantidad de valoraciones — una película con una sola valoración alta no domina el ranking.

In [3]:
popular = weighted_popularity(ratings, movies).head(10)
popular[["title", "count", "mean", "weighted_score"]].to_csv(reports / "popular_top10.csv")
popular[["title", "count", "mean", "weighted_score"]]

,title,count,mean,weighted_score
movieId,,,,
318,"Shawshank Redemption, The (1994)",317,4.429022,4.395194
858,"Godfather, The (1972)",192,4.289062,4.242739
2959,Fight Club (1999),218,4.272936,4.232690
260,Star Wars: Episode IV - A New Hope (1977),251,4.231076,4.197790
50,"Usual Suspects, The (1995)",204,4.237745,4.196846
1221,"Godfather: Part II, The (1974)",129,4.259690,4.195168
527,Schindler's List (1993),220,4.225000,4.187581
1213,Goodfellas (1990),126,4.250000,4.184918
750,Dr. Strangelove or: How I Learned to Stop Worr...,97,4.268041,4.183658


## Paso 3 · Recomendador por contenido (TF-IDF + similitud coseno)

Los géneros se representan con TF-IDF y la recomendación usa similitud coseno. Esto significa
«comparte características registradas», no «le gustará al usuario».

In [4]:
TITLE = "Toy Story (1995)"
content_model = ContentRecommender().fit(movies)
if TITLE not in set(movies["title"]):
    TITLE = movies.sort_values("title").iloc[0]["title"]

content_recs = content_model.recommend(TITLE, 10)
content_recs.to_csv(reports / "content_recommendations.csv", index=False)
print("Recomendaciones de contenido para:", TITLE)
content_recs

Recomendaciones de contenido para: Toy Story (1995)


,movieId,title,genres,content_score
0,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
2,3754,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
4,91355,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
5,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
6,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
7,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
8,136016,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,1.0
9,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0


## Paso 4 · Cierre interpretativo — LAB08 (redacción del estudiante)

**Esta celda es texto, no código — requiere criterio humano, no se completa automáticamente.**
Responda cada punto con base en los resultados reales de arriba:

> *Nota de transparencia:* este cierre fue redactado por la IA a solicitud del estudiante, con base
> únicamente en los resultados reales de las celdas de arriba (sin inventar ningún número). Debe
> leerse y, de ser necesario, ajustarse al criterio propio antes de la entrega final — ver
> `docs/USO_DE_IA.md`.

- **Resultado principal:** el recomendador por contenido (TF-IDF sobre géneros + similitud coseno)
  devuelve, para *Toy Story (1995)*, 10 películas con `content_score = 1.0` exacto — un empate
  perfecto, no una lista ordenada por grados de parecido.
- **Evidencia utilizada:** la tabla de `content_recommendations.csv` (Paso 3): las 10 filas
  (Turbo, The Emperor's New Groove, The Adventures of Rocky and Bullwinkle, Toy Story 2, Asterix
  and the Vikings, Antz, The Wild, Shrek the Third, The Good Dinosaur, Monsters, Inc.) comparten el
  mismo combo de géneros `Adventure|Animation|Children|Comedy|Fantasy` que la película de
  referencia.
- **Qué representa la similitud:** un `content_score` de 1.0 no significa "película casi idéntica"
  en trama, calidad o público — significa que el vector TF-IDF de géneros es idéntico. Es una
  medida de solapamiento de metadatos de género, no de similitud real entre películas.
- **Problema de cold start observado:** `ContentRecommender.recommend(TITLE, ...)` solo necesita el
  título/géneros de una película, sin historial de usuario — por eso resuelve el cold start de
  **ítems nuevos** (una película recién agregada con géneros completos ya es recomendable), pero no
  resuelve el cold start de **usuarios nuevos**: esta celda no personaliza nada por usuario. Ese
  vacío solo se cierra más adelante, en LAB09 (Paso 8), con el fallback de popularidad.
- **Riesgo de sobre-especialización:** el empate exacto en 1.0 entre las 10 recomendaciones es una
  burbuja de género real: el sistema repite el mismo subconjunto de animación infantil y no
  distingue calidad, año o audiencia dentro de ese grupo — el orden entre las 10 es, en la
  práctica, arbitrario.
- **Decisión o siguiente experimento:** el siguiente paso lógico (y el que se prueba en el Paso 7)
  es combinar esta señal de contenido con la señal colaborativa en un sistema híbrido, para romper
  los empates de género con una noción real de calidad/preferencia observada.


## Paso 5 · Corte temporal (evaluación LAB09)

Se reserva la última valoración de cada usuario elegible como prueba (`temporal_leave_one_out`,
el mismo corte que usa `scripts/lab09_hybrid.py`); el resto es entrenamiento.

In [5]:
train, test = temporal_leave_one_out(ratings)
print("Train:", train.shape, " Test (1 por usuario):", test.shape)

Train:

 (100226, 4)  Test (1 por usuario): (610, 4)


## Paso 6 · Factorización matricial (ratings)

RMSE mide qué tan bien el modelo aproxima los ratings retenidos — no garantiza por sí solo un
Top-N más útil; eso se evalúa aparte con métricas de ranking.

In [6]:
start = perf_counter()
mf_model = MatrixFactorization(factors=20, seed=SEED).fit(train, epochs=12)
train_seconds = perf_counter() - start

evaluable = test[test["userId"].isin(mf_model.user_index) & test["movieId"].isin(mf_model.item_index)]
predictions = [mf_model.predict(row.userId, row.movieId) for row in evaluable.itertuples()]
rmse = root_mean_squared_error(evaluable["rating"], predictions)

print(f"RMSE: {rmse:.4f}  |  Entrenamiento: {train_seconds:.2f}s  |  Evaluables: {len(evaluable)}/{len(test)}")

RMSE: 1.0264  |  Entrenamiento: 22.65s  |  Evaluables: 587/610


## Paso 7 · Sistema híbrido (ranking, alpha=0.25 vs. alpha=0.75)

`alpha` controla el peso de la señal colaborativa frente a la de contenido. Se mantiene fija la
semilla, el corte, los factores y las épocas para que la comparación sea controlada — solo
cambia `alpha`.

> Nota de corrección: esta celda incluye un ajuste (`np.asarray(...)`) sobre un bug real que se
> encontró en `scripts/lab09_hybrid.py` — ver `docs/USO_DE_IA.md`.

In [7]:
movie_table = movies.reset_index(drop=True).copy()
movie_table["genres_text"] = movie_table["genres"].str.replace("|", " ", regex=False)
genre_matrix = TfidfVectorizer().fit_transform(movie_table["genres_text"])
movie_row = {int(movie_id): index for index, movie_id in enumerate(movie_table["movieId"])}

popular_train = train.groupby("movieId")["rating"].agg(["mean", "count"]).sort_values(["count", "mean"], ascending=False)
popular_items = popular_train.index.tolist()


def hybrid_recommendations(alpha):
    recommendations = {}
    for user_id in evaluable["userId"].unique():
        seen = set(train.loc[train["userId"].eq(user_id), "movieId"])
        collaborative = mf_model.top_n(user_id, seen, 100)
        if collaborative.empty:
            continue
        collaborative = collaborative.copy()
        collaborative["normalized"] = (
            collaborative["collaborative_score"] - collaborative["collaborative_score"].min()
        ) / max(collaborative["collaborative_score"].max() - collaborative["collaborative_score"].min(), 1e-9)
        history = train.loc[train["userId"].eq(user_id) & train["movieId"].isin(movie_row), ["movieId", "rating"]]
        history_rows = [movie_row[int(item)] for item in history["movieId"]]
        weights = np.clip(history["rating"].to_numpy() - 2.5, 0.1, None)
        profile = np.asarray(genre_matrix[history_rows].multiply(weights[:, None]).sum(axis=0) / weights.sum())
        candidate_rows = [movie_row[int(item)] for item in collaborative["movieId"]]
        content_scores = cosine_similarity(profile, genre_matrix[candidate_rows]).ravel()
        content_min, content_max = content_scores.min(), content_scores.max()
        collaborative["content_score"] = (content_scores - content_min) / max(content_max - content_min, 1e-9)
        collaborative["hybrid_score"] = alpha * collaborative["normalized"] + (1 - alpha) * collaborative["content_score"]
        recommendations[int(user_id)] = collaborative.nlargest(10, "hybrid_score")["movieId"].astype(int).tolist()
    return recommendations


recs_low_alpha = hybrid_recommendations(0.25)
recs_high_alpha = hybrid_recommendations(0.75)

hybrid_comparison = pd.DataFrame([
    {"alpha": 0.25, "hit_rate_at_10": hit_rate_at_k(recs_low_alpha, evaluable, 10), "catalog_coverage": catalog_coverage(recs_low_alpha, len(mf_model.items))},
    {"alpha": 0.75, "hit_rate_at_10": hit_rate_at_k(recs_high_alpha, evaluable, 10), "catalog_coverage": catalog_coverage(recs_high_alpha, len(mf_model.items))},
])
hybrid_comparison.to_csv(reports / "hybrid_alpha_comparison.csv", index=False)
hybrid_comparison

,alpha,hit_rate_at_10,catalog_coverage
0,0.25,0.032368,0.106484
1,0.75,0.037479,0.077518


## Paso 8 · Evaluar tres perfiles de usuario

Usuario con historial amplio, usuario con historial pequeño, y un usuario nuevo que no existe en
el entrenamiento (cold start) — para este último debe usarse el fallback de popularidad, sin
personalizar.

In [8]:
cold_start_items = popular_items[:10]
cold_start_fallback = (
    movies[movies["movieId"].isin(cold_start_items)]
    .set_index("movieId")
    .loc[cold_start_items]
    .join(popular_train[["mean", "count"]])
    .reset_index()
)
cold_start_fallback.to_csv(reports / "cold_start_fallback.csv", index=False)

history_counts = train.groupby("userId").size().sort_values(ascending=False)
user_amplio = int(history_counts.index[0])
user_pequeno = int(history_counts[history_counts >= 5].index[-1]) if (history_counts >= 5).any() else int(history_counts.index[-1])
user_nuevo = int(ratings["userId"].max() + 1000)

print("Usuario con historial amplio:", user_amplio, "->", int(history_counts.loc[user_amplio]), "valoraciones")
print("Usuario con historial pequeno:", user_pequeno, "->", int(history_counts.loc[user_pequeno]), "valoraciones")
print("Usuario nuevo (cold start):", user_nuevo, "-> no existe en el entrenamiento")

Usuario con historial amplio: 414 -> 2697 valoraciones
Usuario con historial pequeno: 53 -> 19 valoraciones
Usuario nuevo (cold start): 1610 -> no existe en el entrenamiento


In [9]:
seen_amplio = set(train.loc[train["userId"].eq(user_amplio), "movieId"])
top_amplio = mf_model.top_n(user_amplio, seen_amplio, 10).merge(movies[["movieId", "title"]], on="movieId")
print("Top-10 colaborativo, historial amplio:")
top_amplio

Top-10 colaborativo, historial amplio:


,movieId,collaborative_score,title
0,2324,4.589133,Life Is Beautiful (La Vita è bella) (1997)
1,1203,4.500309,12 Angry Men (1957)
2,1219,4.357707,Psycho (1960)
3,914,4.328894,My Fair Lady (1964)
4,1230,4.310671,Annie Hall (1977)
5,246,4.286525,Hoop Dreams (1994)
6,1258,4.286293,"Shining, The (1980)"
7,922,4.271964,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)
8,31658,4.261993,Howl's Moving Castle (Hauru no ugoku shiro) (2...
9,92259,4.228366,Intouchables (2011)


In [10]:
seen_pequeno = set(train.loc[train["userId"].eq(user_pequeno), "movieId"])
top_pequeno = mf_model.top_n(user_pequeno, seen_pequeno, 10).merge(movies[["movieId", "title"]], on="movieId")
print("Top-10 colaborativo, historial pequeno:")
top_pequeno

Top-10 colaborativo, historial pequeno:


,movieId,collaborative_score,title
0,858,3.735842,"Godfather, The (1972)"
1,1208,3.711548,Apocalypse Now (1979)
2,1230,3.711483,Annie Hall (1977)
3,63082,3.688449,Slumdog Millionaire (2008)
4,1954,3.687746,Rocky (1976)
5,1213,3.686071,Goodfellas (1990)
6,47,3.685564,Seven (a.k.a. Se7en) (1995)
7,924,3.685142,2001: A Space Odyssey (1968)
8,41285,3.680154,Match Point (2005)
9,593,3.670675,"Silence of the Lambs, The (1991)"


In [11]:
print("Usuario nuevo: el modelo colaborativo no tiene senal (user_id fuera del entrenamiento).")
print("Fallback de popularidad (no personalizado):")
cold_start_fallback[["title", "mean", "count"]]

Usuario nuevo: el modelo colaborativo no tiene senal (user_id fuera del entrenamiento).
Fallback de popularidad (no personalizado):


,title,mean,count
0,Forrest Gump (1994),4.161585,328
1,"Shawshank Redemption, The (1994)",4.428571,315
2,Pulp Fiction (1994),4.191803,305
3,"Matrix, The (1999)",4.192446,278
4,"Silence of the Lambs, The (1991)",4.158845,277
5,Star Wars: Episode IV - A New Hope (1977),4.231076,251
6,Braveheart (1995),4.031646,237
7,Jurassic Park (1993),3.747881,236
8,Terminator 2: Judgment Day (1991),3.963801,221
9,Fight Club (1999),4.272936,218


## Paso 9 · Comparación de costo y utilidad (Green AI, 3 semillas)

Se repite el entrenamiento con 3 semillas distintas, mismos factores y épocas, para ver cuánto
varían RMSE, HitRate@10, cobertura y tiempo de entrenamiento — sin asumir que una sola corrida
representa el comportamiento típico del modelo.

In [12]:
seeds = [7, 42, 123]
pareto_rows = []
for seed in seeds:
    start = perf_counter()
    model_s = MatrixFactorization(factors=20, seed=seed).fit(train, epochs=12)
    seconds = perf_counter() - start
    preds = [model_s.predict(row.userId, row.movieId) for row in evaluable.itertuples()]
    rmse_s = root_mean_squared_error(evaluable["rating"], preds)
    recs_s = {}
    for user_id in evaluable["userId"].unique():
        seen = set(train.loc[train["userId"].eq(user_id), "movieId"])
        recs_s[int(user_id)] = model_s.top_n(user_id, seen, 10)["movieId"].astype(int).tolist()
    pareto_rows.append({
        "seed": seed,
        "rmse": rmse_s,
        "hit_rate_at_10": hit_rate_at_k(recs_s, evaluable, 10),
        "catalog_coverage": catalog_coverage(recs_s, len(model_s.items)),
        "train_seconds": seconds,
    })

pareto = pd.DataFrame(pareto_rows)
pareto.to_csv(reports / "pareto_seeds.csv", index=False)
pareto

,seed,rmse,hit_rate_at_10,catalog_coverage,train_seconds
0,7,1.025865,0.039182,0.068240,21.144853
1,42,1.026418,0.037479,0.074631,20.941741
2,123,1.026084,0.028961,0.067416,21.560612


## Paso 10 · Entorno de ejecución (reproducibilidad)

In [13]:
print("Fecha de ejecucion:", datetime.now().isoformat(timespec="seconds"))
print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("pandas:", pd.__version__, " scikit-learn:", sklearn.__version__, " numpy:", np.__version__)

Fecha de ejecucion: 2026-10-20T22:27:46
Python: 3.12.15
Plataforma: Windows-11-10.0.26300-SP0
pandas: 2.3.3  scikit-learn: 1.9.1  numpy: 2.5.3


## Paso 11 · Cierre interpretativo — LAB09 (redacción del estudiante)

**Esta celda es texto, no código — requiere criterio humano, no se completa automáticamente.**
Responda cada punto con base en los resultados reales de arriba:

> *Nota de transparencia:* este cierre fue redactado por la IA a solicitud del estudiante, con base
> únicamente en los resultados reales de las celdas de arriba (sin inventar ningún número). Debe
> leerse y, de ser necesario, ajustarse al criterio propio antes de la entrega final — ver
> `docs/USO_DE_IA.md`.

- **Resultado de ratings:** la factorización matricial (20 factores, 12 épocas) obtuvo
  RMSE = 1.0264 sobre 587/610 usuarios evaluables (23 quedaron fuera porque su única valoración
  de prueba correspondía a un ítem o usuario ausente del entrenamiento). En una escala de 0.5 a
  5.0 estrellas, eso equivale a un error típico de aproximadamente una estrella.
- **Resultado de ranking:** HitRate@10 fue 0.0324 con alpha=0.25 y 0.0375 con alpha=0.75 — subir el
  peso colaborativo mejoró el acierto del top-10, pero de forma modesta (poco más de medio punto
  porcentual).
- **Cobertura del catálogo:** 10.6% de cobertura con alpha=0.25 frente a 7.8% con alpha=0.75 — más
  peso colaborativo concentra las recomendaciones en un subconjunto más pequeño del catálogo.
- **Comportamiento del usuario frío:** el usuario sintético 1610 (fuera del entrenamiento) no tiene
  ninguna señal colaborativa; el sistema cae al fallback de popularidad suavizada (Forrest Gump,
  The Shawshank Redemption, Pulp Fiction, The Matrix, ...) — una lista genérica, igual para
  cualquier usuario nuevo, pero que al menos siempre devuelve algo en vez de fallar.
- **Configuración seleccionada y evidencia:** se elige **alpha=0.25**. La ganancia de acierto al
  subir a 0.75 es pequeña en términos absolutos (0.0324 → 0.0375), mientras que la pérdida de
  cobertura es proporcionalmente mayor (10.6% → 7.8%, una caída relativa de ~27%). Para un
  recomendador académico donde también importa mostrar variedad del catálogo, ese cambio no
  compensa la concentración adicional.
- **Costo computacional:** cada entrenamiento tomó entre 20.9 y 21.6 segundos en CPU (semillas 7,
  42, 123: 21.14s, 20.94s, 21.56s) — un costo bajo que no es el cuello de botella del proyecto.
  El RMSE se mantuvo estable entre semillas (1.0259–1.0264), pero el HitRate@10 varió más
  (0.0290–0.0392 según semilla), así que debe reportarse como un rango y no como un número único.
- **Riesgo principal y mitigación:** el riesgo principal es la cobertura baja (7.8%–10.6%), que en
  producción se traduciría en una burbuja de popularidad — muchos usuarios verían el mismo
  subconjunto reducido de películas. Mitigación propuesta: monitorear cobertura y HitRate por
  segmento de usuario (historial amplio vs. pequeño vs. nuevo) en vez de solo el promedio global,
  y marcar explícitamente las listas de fallback como "no personalizadas" para no generar una
  falsa expectativa de personalización en usuarios nuevos.


## Paso 12 · Siguientes pasos

1. Corra `uv run pytest -q` para confirmar que los contratos automatizados siguen en verde.
2. Complete `docs/DATASET_CARD.md` y `docs/SYSTEM_CARD.md` con los datos y métricas reales de
   arriba y de `reports/`.
3. Complete a mano los dos cierres interpretativos (Pasos 4 y 11).
4. Con esto, arme el PDF de entrega del Ejercicio 05.